In [2]:
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

# Cargar datos
df = pd.read_csv('../../data/raw/youtoxic_english_1000.csv')  

# Seleccionar solo las columnas necesarias
df_model = df[['Text', 'IsToxic']].copy()
df_model['IsToxic'] = df_model['IsToxic'].astype(int)

# Preprocesamiento
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\r\n|\r|\n', ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words
        and len(t) > 2
    ]
    return ' '.join(tokens)

# Aplicar y guardar
df_model['text_clean'] = df_model['Text'].apply(preprocess)
df_model.to_csv('../../data/processed/comments_processed.csv', index=False)

print(df_model[['Text', 'text_clean', 'IsToxic']].head(3))
print(f"\nGuardado OK — {len(df_model)} filas")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


                                                Text  \
0  If only people would just take a step back and...   
1  Law enforcement is not trained to shoot to app...   
2  \r\nDont you reckon them 'black lives matter' ...   

                                          text_clean  IsToxic  
0  people would take step back make case wasnt an...        0  
1  law enforcement trained shoot apprehend traine...        1  
2  dont reckon black life matter banner held whit...        1  

Guardado OK — 1000 filas


Paso a paso lo que ocurre:

Lowercase — todo a minúsculas
Regex URLs — no había URLs en este, sin cambio
Regex \r\n — eliminó el salto de línea del principio
Regex [^a-z\s] — eliminó las comillas simples ' y el ?
Stopwords — eliminó you, them, by — palabras sin significado semántico
Lematización — lives → life (forma base)
len > 2 — eliminó palabras de 1-2 caracteres

stemming


In [3]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

df_model['text_stemmed'] = df_model['text_clean'].apply(
    lambda x: ' '.join([stemmer.stem(t) for t in x.split()])
)

# Comparar lematización vs stemming
print(df_model[['text_clean', 'text_stemmed']].head(5))

                                          text_clean  \
0  people would take step back make case wasnt an...   
1  law enforcement trained shoot apprehend traine...   
2  dont reckon black life matter banner held whit...   
3  large number people like police officer called...   
4  arab dude absolutely right shot extra time sho...   

                                        text_stemmed  
0  peopl would take step back make case wasnt any...  
1  law enforc train shoot apprehend train shoot k...  
2  dont reckon black life matter banner held whit...  
3  larg number peopl like polic offic call crimin...  
4  arab dude absolut right shot extra time shoot ...  


Data augmentation

In [4]:
# Augmentación por duplicación de clase minoritaria
toxic_df = df_model[df_model['IsToxic'] == 1].sample(
    n=76, random_state=42
)

df_augmented = pd.concat([df_model, toxic_df], ignore_index=True)
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_augmented['IsToxic'].value_counts())
df_augmented.to_csv('../../data/processed/comments_augmented.csv', index=False)

IsToxic
1    538
0    538
Name: count, dtype: int64


In [5]:
from deep_translator import GoogleTranslator
import time

def back_translate(text, mid_lang='es'):
    try:
        # Inglés → Español → Inglés
        translated = GoogleTranslator(source='en', target=mid_lang).translate(text)
        time.sleep(0.5)  # evitar rate limit
        back = GoogleTranslator(source=mid_lang, target='en').translate(translated)
        return back
    except Exception as e:
        return text  # si falla devuelve el original

# Aplicar solo a la clase minoritaria — tóxicos (462 filas)
toxic_df = df_model[df_model['IsToxic'] == 1].copy()

print(f"Traduciendo {len(toxic_df)} comentarios tóxicos...")
toxic_df['text_clean'] = toxic_df['text_clean'].apply(
    lambda x: back_translate(x) if isinstance(x, str) else x
)
toxic_df['augmented'] = 'back_translation'
df_model['augmented'] = 'original'

df_bt = pd.concat([df_model, toxic_df], ignore_index=True)
df_bt = df_bt.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_bt['IsToxic'].value_counts())
print(df_bt['augmented'].value_counts())

# Verificar que funciona
print("\nEjemplo:")
sample = df_model[df_model['IsToxic']==1]['text_clean'].iloc[0]
print(f"Original:   {sample}")
print(f"Traducido:  {back_translate(sample)}")

df_bt.to_csv('../../data/processed/comments_backtranslation.csv', index=False)
print("\nGuardado OK ✅")

Traduciendo 462 comentarios tóxicos...
IsToxic
1    924
0    538
Name: count, dtype: int64
augmented
original            1000
back_translation     462
Name: count, dtype: int64

Ejemplo:
Original:   law enforcement trained shoot apprehend trained shoot kill thank wilson killing punk bitch
Traducido:  law enforcement shoot trained apprehend shoot trained kill thank wilson kill bitch punk

Guardado OK ✅


In [7]:
# Solo traducir las filas necesarias para igualar clases (76 filas)
toxic_df = df_model[df_model['IsToxic'] == 1].sample(n=76, random_state=42).copy()

print(f"Traduciendo {len(toxic_df)} comentarios...")
toxic_df['text_clean'] = toxic_df['text_clean'].apply(
    lambda x: back_translate(x) if isinstance(x, str) else x
)
toxic_df['augmented'] = 'back_translation'
df_model['augmented'] = 'original'

df_bt_balanced = pd.concat([df_model, toxic_df], ignore_index=True)
df_bt_balanced = df_bt_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_bt_balanced['IsToxic'].value_counts())
df_bt_balanced.to_csv('../../data/processed/comments_bt_balanced.csv', index=False)
print("Guardado OK ✅")

Traduciendo 76 comentarios...
IsToxic
1    538
0    538
Name: count, dtype: int64
Guardado OK ✅
